# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Features known at sale time: `channel`, `device_age_years`, `price_eur`, `daily_hours` (registration). `device_serial` is an identifier and is dropped.

Data notes: there is no date column and every `device_serial` is unique, so there is no time order or repeated entity to respect. Rows are treated as independent and the split is random and stratified. If the model will be used on later sales, this assumes drift is small; that cannot be checked with this data.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
df = pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
print(df.claim_within_year.mean())
df.describe()

## 1. Test set set aside first (20%, stratified), untouched until the final step

In [ ]:
FEATURES = ['channel','device_age_years','price_eur','daily_hours']
TARGET = 'claim_within_year'
train, test = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=0)
Xtr, ytr = train[FEATURES], train[TARGET]
Xte, yte = test[FEATURES], test[TARGET]
print(len(train), len(test), ytr.mean(), yte.mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (DummyClassifier, prior).
2. `logreg_C*`: logistic regression, one-hot channel + standardised numerics, C in {0.1, 1, 10}.
3. `hgb_d*`: histogram gradient boosting, max_depth in {2, 3}, learning_rate 0.05, max_iter 200.

Selection metric: cross-validated ROC AUC (threshold-free). The decision threshold is then chosen on out-of-fold predictions to maximise F1. All preprocessing lives inside the Pipeline, so it is fit within each fold.

In [ ]:
def lin_pre():
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['channel']),
                              ('num', StandardScaler(), ['device_age_years','price_eur','daily_hours'])])
def hgb_pre():
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['channel']),
                              ('num', 'passthrough', ['device_age_years','price_eur','daily_hours'])])
candidates = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]:
    candidates[f'logreg_C{C_}'] = Pipeline([('p',lin_pre()),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,3]:
    candidates[f'hgb_d{d}'] = Pipeline([('p',hgb_pre()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))])
list(candidates)

## 3. Cross-validation on training rows only (5-fold stratified, 3 seeds)

In [ ]:
def oof(model, seed):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    return cross_val_predict(model, Xtr, ytr, cv=cv, method='predict_proba')[:,1]
oofs = {n:[oof(m,s) for s in range(3)] for n,m in candidates.items()}
rows=[]
for n,ps in oofs.items():
    a=[roc_auc_score(ytr,p) for p in ps]; b=[average_precision_score(ytr,p) for p in ps]
    rows.append((n,np.mean(a),np.std(a),np.mean(b)))
cvres = pd.DataFrame(rows,columns=['model','cv_auc','auc_sd_over_seeds','cv_ap']).sort_values('cv_auc',ascending=False)
cvres

In [ ]:
best = cvres.iloc[0].model
print(best)
# threshold chosen on out-of-fold predictions of the chosen model (mean over seeds)
p = np.mean(oofs[best],axis=0)
grid = np.linspace(0.02,0.6,59)
f1s = [f1_score(ytr, p>=t) for t in grid]
thr = float(grid[int(np.argmax(f1s))])
print('threshold', thr, 'cv F1', max(f1s), 'precision', precision_score(ytr,p>=thr), 'recall', recall_score(ytr,p>=thr))
print('F1 if everyone is flagged:', f1_score(ytr, np.ones(len(ytr))))

## 4. Decision (written before the test set is scored)
The chosen model and threshold are those printed above: the candidate with the highest cross-validated AUC, with the threshold that maximises out-of-fold F1. Nothing is changed after this point.

In [ ]:
final = candidates[best].fit(Xtr, ytr)
pt = final.predict_proba(Xte)[:,1]
base = candidates['baseline'].fit(Xtr,ytr).predict_proba(Xte)[:,1]
pred = pt>=thr
print('TEST ROC AUC', roc_auc_score(yte,pt), '(baseline', roc_auc_score(yte,base),')')
print('TEST avg precision', average_precision_score(yte,pt), '(prevalence', yte.mean(),')')
print('TEST F1', f1_score(yte,pred), 'precision', precision_score(yte,pred), 'recall', recall_score(yte,pred))
print(confusion_matrix(yte,pred))
rng=np.random.default_rng(0); aucs=[]
for _ in range(1000):
    i=rng.integers(0,len(yte),len(yte))
    if yte.iloc[i].nunique()>1: aucs.append(roc_auc_score(yte.iloc[i],pt[i]))
print('AUC 95% bootstrap CI', np.percentile(aucs,[2.5,97.5]))

## 5. Conclusion
The test numbers above were scored once. Read AUC against 0.5 and its CI. Caveats: no time column, so no out-of-time validation; only four features.